In [2]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install optuna

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 76.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [128]:
import torch
from torch import nn
import pandas as pd
import random
import optuna
from torch.optim.lr_scheduler import OneCycleLR

In [129]:
df = pd.read_csv("gru_reversal_data.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(len(df))

cuda
79994


In [130]:
df['source'] = df['source'].apply(lambda x: eval(x))
df['context_vector'] = df['context_vector'].apply(lambda x: eval(x))

In [131]:
df_sorted = df.sort_values(by='source', key=lambda col: col.map(len))

In [132]:
model_size = len(df_sorted['context_vector'].iloc[0])
print(model_size)   

128


In [133]:
# Max batch size possible without padding
batch_size = 128

In [134]:
left_role = 0
right_role = 1
def right_recurse(lst):
    """
    Takes a list of 1D tensors (one per position) of shape (batch_size,)
    and returns a batched nested structure.
    """
    if not lst:
        return []
    
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), left_role, device = device)] for x in lst]
    
    acc = [tensors[-1]]
    for t in reversed(tensors[:-1]):
        acc = [t, [acc, torch.full((batch_size,), right_role, device = device)]]
    return acc

def left_recurse( lst):
    """
    Takes a list of 1D tensors (one per position) of shape (batch_size,)
    and returns a batched nested structure.
    """
    if not lst:
        return []
    
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), right_role, device = device)] for x in lst]
    
    acc = [tensors[0]]
    for t in tensors[1:]:
        acc = [[acc, torch.full((batch_size,), left_role, device = device)], t]
    return acc

In [135]:
def collate_lists(*lists):
    """
    Takes multiple lists of integers of the same length and returns
    a list of tensors, one per position across the batch.
    
    Example: collate_lists([8,3,6,2], [1,5,7,4], [2,2,1,6])
    Returns: [tensor([8,1,2]), tensor([3,5,7]), tensor([6,7,1]), tensor([2,4,6])]
    """
    assert len(set(len(l) for l in lists)) == 1, "All lists must be the same length"
    
    return [torch.tensor([lst[i] for lst in lists]) for i in range(len(lists[0]))]

In [136]:
sources = df_sorted["source"].tolist()
targets = df_sorted["context_vector"].tolist()

# --- Bucket batching: group sequences by length, batch within each bucket ---
from collections import defaultdict

buckets = defaultdict(list)  # seq_len -> list of (source, target) pairs

for src, trg in zip(sources, targets):
    buckets[len(src)].append((src, trg))

del buckets[9]
del buckets[10]
del buckets[11]

In [137]:
src_batches = []
trg_batches = []

for seq_len in sorted(buckets.keys()):
    pairs = buckets[seq_len]
    # Batch within this length bucket; last batch may be smaller — that's okay
    for i in range(0, len(pairs), batch_size):
        # If i + batch_size is out of bounds, it just goes to the end of the list
        chunk = pairs[i:i + batch_size]
        src_batch = [p[0] for p in chunk]
        trg_batch = [p[1] for p in chunk]

        # collate_lists expects *lists unpacked, where each list is one sequence's tokens
        # so we unpack the source sequences — gives one tensor per position across the batch
        collated = collate_lists(*src_batch)     # list of tensors, each (batch,)
        # nested   = right_recurse(collated)
        nested   = left_recurse(collated)

        src_batches.append(nested)
        trg_batches.append(torch.stack([torch.tensor(v, dtype=torch.float32) for v in trg_batch]))  # (batch, model_size)

# 80/10/10 split at batch level
n       = len(src_batches)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
# test: remainder

combined = list(zip(src_batches, trg_batches))
random.shuffle(combined)
src_batches, trg_batches = zip(*combined)

train_src, train_trg = src_batches[:n_train],               trg_batches[:n_train]
val_src,   val_trg   = src_batches[n_train:n_train+n_val],  trg_batches[n_train:n_train+n_val]
test_src,  test_trg  = src_batches[n_train+n_val:],         trg_batches[n_train+n_val:]

In [138]:
def get_recursive_batch_size(x):
    if isinstance(x[0], list):
        return get_recursive_batch_size(x[0])
    return x[0].shape[0]
    
class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        cur_batch_size = get_recursive_batch_size(x)
        
        result = torch.zeros(cur_batch_size, self.filler_size, self.role_size, device=self.w.device)
    
        for pair in x:
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            role = self.roles(role)
            result += torch.einsum("bf,br->bfr", filler, role)
        
        result = result.view(cur_batch_size, -1)
        result = torch.einsum("ij,bj->bi", self.w, result) 
        return result

In [139]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = trg[i].to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch)
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [140]:
def objective(trial):
    num_epochs = 50

    model = RTPREncode(12, 2, 64, 64, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch)
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        print(f"Validation Loss: {val_loss}")

    return val_loss
    

In [141]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=8)
print("Best Hyperparameters:", study.best_params) 

[I 2026-06-13 17:02:13,917] A new study created in memory with name: no-name-409c779d-5062-4ea4-9835-f5b335389a6e


Epoch 1 / 50
Train Loss: 0.03438913442526653
Validation Loss: 0.02132488803890271
Epoch 2 / 50
Train Loss: 0.01745523520816521
Validation Loss: 0.012306772100810822
Epoch 3 / 50
Train Loss: 9.573376488646456
Validation Loss: 0.024334155787260104
Epoch 4 / 50
Train Loss: 0.01406297992891187
Validation Loss: 0.005524476876673408
Epoch 5 / 50
Train Loss: 0.021793593433062742
Validation Loss: 0.026557775954596508
Epoch 6 / 50
Train Loss: 0.02153886099951045
Validation Loss: 0.01973823009011073
Epoch 7 / 50
Train Loss: 29596.089435481485
Validation Loss: 0.039477973412244745
Epoch 8 / 50
Train Loss: 0.02307362051734072
Validation Loss: 0.00932505064142438
Epoch 9 / 50
Train Loss: 0.007334841644038464
Validation Loss: 0.0034544122202369645
Epoch 10 / 50
Train Loss: 0.009201899049640486
Validation Loss: 0.013128310585251221
Epoch 11 / 50
Train Loss: 0.012585497162435323
Validation Loss: 0.019161671412010223
Epoch 12 / 50
Train Loss: 0.008839142172535062
Validation Loss: 0.0032052279939540685


[I 2026-06-13 17:03:04,066] Trial 0 finished with value: 0.0015341959019096042 and parameters: {'lr': 5.508375524321716e-05, 'grad_norm': 5.846550819980682, 'scheduler_max': 0.00660159456081365}. Best is trial 0 with value: 0.0015341959019096042.


Train Loss: 0.0016807652438937384
Validation Loss: 0.0015341959019096042
Epoch 1 / 50
Train Loss: 0.18483778687229582
Validation Loss: 0.05522019750414751
Epoch 2 / 50
Train Loss: 0.05023709958668347
Validation Loss: 0.044592767189710565
Epoch 3 / 50
Train Loss: 0.04172729674702997
Validation Loss: 0.03583293470243613
Epoch 4 / 50
Train Loss: 0.03504052489497669
Validation Loss: 0.029511659477765743
Epoch 5 / 50
Train Loss: 0.028402893703169882
Validation Loss: 0.02223421924580366
Epoch 6 / 50
Train Loss: 0.0207405768714513
Validation Loss: 0.017303593671665743
Epoch 7 / 50
Train Loss: 0.014309088092973562
Validation Loss: 0.010433447714417409
Epoch 8 / 50
Train Loss: 0.009699986205038846
Validation Loss: 0.007446652194723869
Epoch 9 / 50
Train Loss: 0.006919102065538051
Validation Loss: 0.007527306652030883
Epoch 10 / 50
Train Loss: 0.005087384805795114
Validation Loss: 0.003763743461324618
Epoch 11 / 50
Train Loss: 0.004467037753320661
Validation Loss: 0.0031718949739558576
Epoch 12 

[I 2026-06-13 17:03:54,029] Trial 1 finished with value: 0.0015233389060132396 and parameters: {'lr': 0.00021209244430993974, 'grad_norm': 5.3326047943811785, 'scheduler_max': 0.00011176209442110913}. Best is trial 1 with value: 0.0015233389060132396.


...

Train Loss: 0.01287140957581081
Epoch 1 / 50
Train Loss: 0.07822888629261855
Validation Loss: 0.03100029689570268
Epoch 2 / 50
Train Loss: 0.025004790366217968
Validation Loss: 0.016120352662908725
Epoch 3 / 50
Train Loss: 0.012616478830611535
Validation Loss: 0.007718518400230469
Epoch 4 / 50
Train Loss: 0.0061423870353740015
Validation Loss: 0.004827787741445578
Epoch 5 / 50
Train Loss: 0.0052944158921670766
Validation Loss: 0.004335518472660811
Epoch 6 / 50
Train Loss: 0.009355950710930191
Validation Loss: 0.018013386127467338
Epoch 7 / 50
Train Loss: 0.012570274456190599
Validation Loss: 0.0059657089639073
Epoch 8 / 50
Train Loss: 0.007083659492593235
Validation Loss: 0.004867673081417496
Epoch 9 / 50
Train Loss: 0.005673732321225937
Validation Loss: 0.004710281643873224
Epoch 10 / 50


[I 2026-06-13 17:06:41,790] Trial 6 pruned. 


Train Loss: 0.006116544250191255
Epoch 1 / 50


[I 2026-06-13 17:06:42,795] Trial 7 pruned. 


Train Loss: 0.11330309009570984
Best Hyperparameters: {'lr': 0.00014983602849501972, 'grad_norm': 4.15420456549202, 'scheduler_max': 0.0005058999458382887}


In [142]:
pruned_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Study statistics: 
  Number of finished trials:  8
  Number of pruned trials:  3
  Number of complete trials:  5
Best trial:
  Value:  0.0015052439068826155
  Params: 
    lr: 0.00014983602849501972
    grad_norm: 4.15420456549202
    scheduler_max: 0.0005058999458382887


In [143]:
# Train with optimized hyperparams
num_epochs = 50

model = RTPREncode(13, 2, 64, 64, model_size).to(device)
learning_rate = 0.00014983602849501972
norm_val = 4.15420456549202
max_rate = 0.0005058999458382887

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = train_trg[i].to(device)
            optimizer.zero_grad()

            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch)
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 50
Train Loss: 0.08529100792518088
Validation Loss: 0.04256564646195143
Epoch 2 / 50
Train Loss: 0.037986162480465165
Validation Loss: 0.029106756863303673
Epoch 3 / 50
Train Loss: 0.02538250782971929
Validation Loss: 0.0162431062796177
Epoch 4 / 50
Train Loss: 0.013041894968931272
Validation Loss: 0.008149039179373246
Epoch 5 / 50
Train Loss: 0.007252465873499917
Validation Loss: 0.004449538910427155
Epoch 6 / 50
Train Loss: 0.005204411806348877
Validation Loss: 0.004992638690731464
Epoch 7 / 50
Train Loss: 0.00395961653939478
Validation Loss: 0.0038596123922616243
Epoch 8 / 50
Train Loss: 0.00705998822576574
Validation Loss: 0.00906065619813326
Epoch 9 / 50
Train Loss: 0.009758481924914441
Validation Loss: 0.0104136227701719
Epoch 10 / 50
Train Loss: 0.005201210421332081
Validation Loss: 0.00334045991826898
Epoch 11 / 50
Train Loss: 32.87043495537716
Validation Loss: 0.013952732850343753
Epoch 12 / 50
Train Loss: 0.012769285898661584
Validation Loss: 0.007439904285069459
Ep

In [108]:
src_batch = train_src[0]
trg_batch = train_trg[0].to(device)
# print(src_batch)

encoding = model(src_batch)
# print(encoding)
output = torch.matmul(encoding, model.wo.T)
print(output)
print(trg_batch)

tensor([[ 1.7362e-01, -9.4500e-01, -1.6843e-01,  ..., -6.4011e-02,
          2.3812e-01, -2.8649e-01],
        [-2.9604e-01, -9.3552e-01,  2.2918e-01,  ..., -1.2972e-01,
         -1.6223e-01,  3.4480e-06],
        [-2.1474e-01, -9.5269e-01, -2.0856e-01,  ..., -1.0438e-01,
          3.4197e-01, -3.0757e-01],
        ...,
        [-3.8752e-01, -9.5014e-01,  3.8784e-01,  ..., -1.0011e-01,
          2.5015e-02, -8.5119e-02],
        [-3.0153e-01, -9.4370e-01,  1.9075e-01,  ..., -7.4498e-02,
         -5.3285e-02, -3.1775e-02],
        [ 2.4095e-01, -9.3430e-01, -9.4951e-02,  ..., -3.6815e-02,
         -3.4615e-01, -2.1193e-01]], device='cuda:0', grad_fn=<MmBackward0>)
tensor([[-0.0638, -0.9462,  0.0553,  ..., -0.0646,  0.0701, -0.1794],
        [-0.4610, -0.9366,  0.3227,  ..., -0.1458, -0.0717, -0.2644],
        [-0.2689, -0.9492, -0.3162,  ..., -0.1007,  0.2971, -0.6056],
        ...,
        [-0.4779, -0.9455,  0.3729,  ..., -0.0948,  0.0909, -0.2722],
        [-0.2504, -0.9386,  0.0114,

In [144]:
torch.save(model.state_dict(), "gru_left_reversal.pt")